# 第 04 章：高级工具调用 (Smart Tooling)

根据讲义，本章你将掌握以下核心能力：
- **Pydantic v2 契约建模**：建立坚不可摧的参数校验层。
- **运行时上下文注入**：利用 `InjectedState` 实现安全权限透传。
- **防御式编程实操**：应对小模型的参数生成幻觉。
- **揭秘调度者**：亲手实现一个“零魔法”的工具自动调度器。

## 1. 契约化建模：Pydantic 约束
通过 Pydantic 显式定义 `args_schema`。

In [52]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
import json

class SearchArgs(BaseModel):
    query: str = Field(description="搜索关键词")
    limit: int = Field(default=5, description="结果数量", ge=1, le=10)

@tool(args_schema=SearchArgs)
def smart_search(query: str, limit: int = 5):
    """增强型搜索工具。"""
    return f"[执行搜索] 关键词: {query}, 数量上限: {limit}"

## 2. 安全穿透：使用 InjectedState 注入上下文

In [53]:
from typing import Annotated
from langgraph.prebuilt import InjectedState

@tool
def secure_delete(item_id: str, user_id: Annotated[str, InjectedState("user_id")]):
    """
    安全删除指定项目。LLM 严禁生成 user_id，该字段由系统自动注入。
    """
    return f"[审计成功] 用户 {user_id} 已发起对项目 {item_id} 的删除操作。"

## 3. 重点：揭秘调度者 (The Dispatcher)
仔细观察下方的 `manual_dispatch` 逻辑。**请确保运行该 Cell**，以便下方实验能正常调用它。

In [56]:
def manual_dispatch(ai_msg: AIMessage, state: dict):
    """
    模拟逻辑：读取意图 -> 寻找工具 -> 装载状态 -> 真正执行。
    """
    results = []
    # 定义我们支持的所有本地工具映射
    # 注意：在 Lab 中我们会用到 file_operator
    def dummy_op(file_name, action, user_id): return f"Dummy OP for {file_name}"
    
    for tool_call in ai_msg.tool_calls:
        t_name = tool_call["name"]
        args = tool_call["args"].copy()
        
        # 只有特定的工具才需要注入 user_id
        if t_name in ["secure_delete", "file_operator"]:
            args["user_id"] = state["current_user_id"] # 魔法发生点
            
        # 动态分发执行
        if t_name == "secure_delete":
            res = secure_delete.invoke(args)
            results.append(ToolMessage(content=res, tool_call_id=tool_call["id"]))
        elif t_name == "file_operator":
            # 注意：file_operator 在下方 Lab 中定义
            # 这里我们使用 global 引用，确保能找到下方生成的函数
            res = globals()["file_operator"].invoke(args)
            results.append(ToolMessage(content=res, tool_call_id=tool_call["id"]))
            
    return results

print("调度器 (Dispatcher) 已就绪，已包含 file_operator 路由。")

调度器 (Dispatcher) 已就绪，已包含 file_operator 路由。


## 4. 实战：智能文件权限管理系统
运行此单元格，体验安全拦截效果。确保已运行上方的所有单元格。

In [57]:
ACL = {
    "admin_9527": ["readme.md", "secret.txt"],
    "guest_001": ["readme.md"]
}

@tool
def file_operator(file_name: str, action: str, user_id: Annotated[str, InjectedState("user_id")]):
    """文件操作器。支持 action 为 'read' 或 'delete'。"""
    print(f"--- 正在审计分析: 用户 {user_id} 发起的 {action} 请求 ---")
    allowed_files = ACL.get(user_id, [])
    if file_name not in allowed_files:
        return f"错误：用户 {user_id} 对 {file_name} 权限不足。"
    return f"成功：已完成对 {file_name} 的 {action} 操作。"

# 模拟 AI 生成的意想：读取 secret.txt
mock_intent = AIMessage(
    content="",
    tool_calls=[{"name": "file_operator", "args": {"file_name": "secret.txt", "action": "read"}, "id": "lab_id"}]
)

print("--- 场景 A: Guest 访客视角 ---")
try:
    res_guest = manual_dispatch(mock_intent, {"current_user_id": "guest_001"})
    print(res_guest[0].content)
except Exception as e:
    print(f"发生异常: {e}")

print("\n--- 场景 B: Admin 管理员视角 ---")
try:
    res_admin = manual_dispatch(mock_intent, {"current_user_id": "admin_9527"})
    print(res_admin[0].content)
except Exception as e:
    print(f"发生异常: {e}")

--- 场景 A: Guest 访客视角 ---
--- 正在审计分析: 用户 guest_001 发起的 read 请求 ---
错误：用户 guest_001 对 secret.txt 权限不足。

--- 场景 B: Admin 管理员视角 ---
--- 正在审计分析: 用户 admin_9527 发起的 read 请求 ---
成功：已完成对 secret.txt 的 read 操作。
